<a href="https://colab.research.google.com/github/minkyu156123-crypto/bigdata-project/blob/main/05_%EB%8D%B0%EC%9D%B4%ED%84%B0%ED%86%B5%ED%95%A9_%EB%A7%88%EC%8A%A4%ED%84%B0%ED%85%8C%EC%9D%B4%EB%B8%94%EC%83%9D%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  영화 1-4주차 관객수 + 네이버 검색 + 유튜브 반응 기반 중형 영화 흥행 지속성 예측 — 데이터 통합

##  본 노트북의 목적

KOFIC 흥행 데이터, 네이버 검색량(단독·기대·검증), 유튜브 예고편 통계를 **하나의 마스터 테이블**로 통합합니다.

##  사용 데이터 (6개 파일)

| 파일 | 출처 | 행 수 | 핵심 정보 |
|---|---|---|---|
| `kofic_최종_영화_드롭률_데이터셋.xlsx` | KOFIC API | 387행 | 1~4주차 주말관객 (패널) |
| `movie_드롭률_분석.xlsx` | 수정 전 파일 | 131행 | 스크린·공휴일·분류 |
| `naver_검색량_단독수집__1_.xlsx` | 네이버 데이터랩 | 1,950행 | 영화명 단독 검색지수 |
| `naver_검색량_검증형수집__1_.xlsx` | 네이버 데이터랩 | 1,002행 | 후기·리뷰·평점·관람평 |
| `naver_검색량_기대형수집.xlsx` | 네이버 데이터랩 | 974행 | 예고편·시사회·개봉일·출연 |
| `youtube_수집_최종_수정완료.xlsx` | YouTube Data API | 130행 | 예고편 조회수·좋아요·댓글수 |

##  출력

- `master_dataset_전체.xlsx` (130편 × 33개 변수)
- `master_dataset_중형.xlsx` (중형 102편만 추출)

##  작업 흐름

```

## 1️ 데이터 파일 업로드

###  작업 내용
구글 코랩에 6개 엑셀 파일을 한 번에 업로드합니다. 파일명에 들어있는 키워드(`kofic`, `movie 분석`, `단독`, `검증`, `기대`, `youtube`)로 자동 인식하여 매핑합니다.



## 2️. 각 데이터에서 핵심 변수 추출

###  작업 내용
6개 데이터 각각에서 분석에 사용할 변수를 계산하여 추출합니다.

###  변수 설계 원칙

| 데이터 | 추출 변수 | 설계 이유 |
|---|---|---|
| **KOFIC 패널** | 1~4주차 주말관객 | 흥행 지속성 측정 (★ 본 분석의 타겟 영역) |
| **분석 파일** | 분류, 스크린수, 공휴일 | 통제변수 + 이미 분류된 5개 카테고리 활용 |
| **단독 검색** | 사전평균, 사후평균, 사후/사전 비율 | 영화 전반 관심도의 시점 변화 |
| **기대 검색** | 기대_강도 | 안 본 사람의 정보 탐색 강도 |
| **검증 검색** | 검증_강도, 검증_지속력 | 본 사람의 평가 검색 + 후반 지속성 |
| **유튜브** | log_조회수, 일평균조회수, 좋아요·댓글 비율 | 누적값을 시간 정규화 + 호감도·관여도 |

###  핵심 설계: 검증형 vs 기대형 분리
- **기대형**: 개봉 전 ~ 개봉일 (안 본 사람의 정보 탐색)
- **검증형**: 개봉일 ~ +7일 (본 사람의 평가 활동)
- 두 시점 데이터의 비율 = **입소문 강도** 지표

###  결측치 처리 방침
- 검증·기대 데이터 없는 영화 (5편): **0으로 채움** (검색 활동이 발생하지 않은 것 자체가 정보)
- 3·4주차 관객 결측 (3편): NaN 유지 (4주차 생존비율 계산 시만 0으로 처리)

In [1]:
# ============================================================
# 셀 1. 라이브러리 + 6개 데이터 파일 업로드 (진단 강화 버전)
# ============================================================

import pandas as pd
import numpy as np
import re
from google.colab import files

print("="*60)
print(" 6개 데이터 파일을 한 번에 업로드해주세요")
print("="*60)
print("필요한 파일:")
print("  1. kofic_최종_영화_드롭률_데이터셋.xlsx")
print("  2. movie_드롭률_분석.xlsx")
print("  3. naver_검색량_단독수집__1_.xlsx")
print("  4. naver_검색량_검증형수집__1_.xlsx")
print("  5. naver_검색량_기대형수집.xlsx")
print("  6. youtube_수집_최종_수정완료.xlsx")
print()
print(" Ctrl(Cmd) 누른 채로 6개 파일 동시 선택")
print()

uploaded = files.upload()
print(f"\n {len(uploaded)}개 파일 업로드 완료")

#  업로드된 파일 목록 모두 출력
print(f"\n 업로드된 파일명 목록:")
for fname in uploaded.keys():
    print(f"   - {fname}")

def find_file(pattern, file_dict):
    for fname in file_dict.keys():
        if re.search(pattern, fname, re.IGNORECASE):
            return fname
    return None

# 각 파일 매칭 시도
kofic_panel_file = find_file(r'kofic.*드롭률|kofic.*데이터셋', uploaded)
analysis_file = find_file(r'movie.*분석|드롭률_분석', uploaded)
solo_file = find_file(r'단독', uploaded)
verif_file = find_file(r'검증', uploaded)
expect_file = find_file(r'기대', uploaded)
youtube_file = find_file(r'youtube', uploaded)

print(f"\n 자동 매칭 결과:")
print(f"   KOFIC 패널 (1~4주차): {kofic_panel_file}")
print(f"   분류·스크린·공휴일:   {analysis_file}")
print(f"   단독검색:             {solo_file}")
print(f"   검증검색:             {verif_file}")
print(f"   기대검색:             {expect_file}")
print(f"   유튜브:               {youtube_file}")

#  매칭 실패한 파일 진단
missing = []
if kofic_panel_file is None: missing.append("KOFIC 패널 (kofic + 드롭률 또는 데이터셋)")
if analysis_file is None: missing.append("분류 분석 (movie + 분석)")
if solo_file is None: missing.append("단독 검색")
if verif_file is None: missing.append("검증 검색")
if expect_file is None: missing.append("기대 검색")
if youtube_file is None: missing.append("유튜브 (youtube)")

if missing:
    print(f"\n 매칭 실패한 파일:")
    for m in missing:
        print(f"   - {m}")
    print(f"\n 해결 방법:")
    print(f"   1. 위 '업로드된 파일명 목록'에 해당 파일이 있는지 확인")
    print(f"   2. 파일명에 위 키워드가 포함되어 있는지 확인")
    print(f"   3. 파일이 빠졌다면 셀 1을 다시 실행하여 재업로드")
else:
    print(f"\n 6개 파일 모두 매핑됨, 셀 2로 진행하세요")

 6개 데이터 파일을 한 번에 업로드해주세요
필요한 파일:
  1. kofic_최종_영화_드롭률_데이터셋.xlsx
  2. movie_드롭률_분석.xlsx
  3. naver_검색량_단독수집__1_.xlsx
  4. naver_검색량_검증형수집__1_.xlsx
  5. naver_검색량_기대형수집.xlsx
  6. youtube_수집_최종_수정완료.xlsx

 Ctrl(Cmd) 누른 채로 6개 파일 동시 선택



Saving youtube_수집_최종_수정완료.xlsx to youtube_수집_최종_수정완료.xlsx
Saving kofic_최종_영화 드롭률 데이터셋.xlsx to kofic_최종_영화 드롭률 데이터셋.xlsx
Saving movie_드롭률_분석.xlsx to movie_드롭률_분석.xlsx
Saving naver_검색량_검증형수집 (1).xlsx to naver_검색량_검증형수집 (1).xlsx
Saving naver_검색량_기대형수집.xlsx to naver_검색량_기대형수집.xlsx
Saving naver_검색량_단독수집 (1).xlsx to naver_검색량_단독수집 (1).xlsx

 6개 파일 업로드 완료

 업로드된 파일명 목록:
   - youtube_수집_최종_수정완료.xlsx
   - kofic_최종_영화 드롭률 데이터셋.xlsx
   - movie_드롭률_분석.xlsx
   - naver_검색량_검증형수집 (1).xlsx
   - naver_검색량_기대형수집.xlsx
   - naver_검색량_단독수집 (1).xlsx

 자동 매칭 결과:
   KOFIC 패널 (1~4주차): kofic_최종_영화 드롭률 데이터셋.xlsx
   분류·스크린·공휴일:   movie_드롭률_분석.xlsx
   단독검색:             naver_검색량_단독수집 (1).xlsx
   검증검색:             naver_검색량_검증형수집 (1).xlsx
   기대검색:             naver_검색량_기대형수집.xlsx
   유튜브:               youtube_수집_최종_수정완료.xlsx

 6개 파일 모두 매핑됨, 셀 2로 진행하세요


In [2]:
# ============================================================
# 셀 1.5. 한국 공휴일 자동 계산 + 1·2주차 검증 + 3·4주차 추가
# ============================================================

# holidays 라이브러리 설치 (한국 공휴일 자동 제공)
!pip install holidays --quiet
import holidays
from datetime import timedelta

# ============================================================
# [1] 한국 공휴일 리스트 만들기 (2023~2026)
# ============================================================
kr_holidays = holidays.KR(years=range(2023, 2027))

# 임시공휴일 추가 (라이브러리에 빠진 것)
extra_holidays = {
    '2024-04-10': '제22대 국회의원 선거',
    '2024-10-01': '국군의 날 임시공휴일',
    # 필요시 본인이 추가
}

# 통합 공휴일 set (날짜 객체로)
all_holidays = set(kr_holidays.keys())
for date_str in extra_holidays.keys():
    all_holidays.add(pd.to_datetime(date_str).date())

print(f" 한국 공휴일 로드: {len(all_holidays)}일")
print(f"   2024년 임시공휴일 추가: {list(extra_holidays.values())}")


# ============================================================
# [2] 분석 파일 읽기 + 1주차 금요일 계산
# ============================================================
analysis = pd.read_excel(analysis_file, sheet_name='전체_드롭률')

def parse_first_friday(row):
    """1주차_주말일자 + 연도로부터 1주차 금요일 datetime 생성"""
    weekend_str = str(row['1주차_주말일자'])  # "01/06~01/08"
    year = int(row['연도'])
    open_date = pd.to_datetime(row['개봉일'])

    # MM/DD 추출
    start_mmdd = weekend_str.split('~')[0]  # "01/06"
    month, day = map(int, start_mmdd.split('/'))

    # 1주차가 개봉 연도인지 다음 연도인지 판단 (연말 개봉작 처리)
    candidate = pd.Timestamp(year=year, month=month, day=day)
    if candidate < open_date:
        candidate = pd.Timestamp(year=year+1, month=month, day=day)

    return candidate

analysis['1주차_금요일'] = analysis.apply(parse_first_friday, axis=1)

# ============================================================
# [3] 각 주차별 공휴일 카운트 함수
# ============================================================
def count_weekend_holidays(friday_date, holidays_set):
    """그 주 금/토/일 중 공휴일 일수"""
    fri = friday_date.date()
    sat = (friday_date + timedelta(days=1)).date()
    sun = (friday_date + timedelta(days=2)).date()
    return sum(d in holidays_set for d in [fri, sat, sun])

# ============================================================
# [4] 1~4주차 모두 자동 계산
# ============================================================
for week in range(1, 5):
    col_name = f'auto_{week}주차_공휴일'
    days_offset = (week - 1) * 7

    analysis[col_name] = analysis['1주차_금요일'].apply(
        lambda fri: count_weekend_holidays(fri + timedelta(days=days_offset), all_holidays)
    )

print(f" 자동 계산 완료: 1~4주차 공휴일")


# ============================================================
# [5]  검증: 자동 계산 vs 본인 수동 계산 비교
# ============================================================
print(f"\n{'='*60}")
print(f" 검증: 자동 계산 vs 수동 계산 비교")
print(f"{'='*60}")

# 1주차 비교
match_w1 = (analysis['auto_1주차_공휴일'] == analysis['1주차_공휴일수_주말']).sum()
total = len(analysis)
print(f"\n1주차 일치: {match_w1}/{total} ({match_w1/total*100:.1f}%)")

# 2주차 비교
match_w2 = (analysis['auto_2주차_공휴일'] == analysis['2주차_공휴일수_주말']).sum()
print(f"2주차 일치: {match_w2}/{total} ({match_w2/total*100:.1f}%)")

# 불일치 영화 찾기
mismatches = analysis[
    (analysis['auto_1주차_공휴일'] != analysis['1주차_공휴일수_주말']) |
    (analysis['auto_2주차_공휴일'] != analysis['2주차_공휴일수_주말'])
]

if len(mismatches) > 0:
    print(f"\n 불일치 영화 {len(mismatches)}편:")
    for _, r in mismatches.head(10).iterrows():
        print(f"   [{r['연도']}] {r['영화명']}")
        print(f"      1주차: 자동={r['auto_1주차_공휴일']} vs 수동={r['1주차_공휴일수_주말']}")
        print(f"      2주차: 자동={r['auto_2주차_공휴일']} vs 수동={r['2주차_공휴일수_주말']}")
else:
    print(f"\n 1·2주차 모두 100% 일치 → 자동 계산 신뢰 가능")
    print(f"   3·4주차도 자동 계산 결과를 사용합니다.")


# ============================================================
# [6] 분포 확인 — 3·4주차 공휴일이 얼마나 분포하는가
# ============================================================
print(f"\n 자동 계산 결과 분포:")
for week in range(1, 5):
    col = f'auto_{week}주차_공휴일'
    dist = analysis[col].value_counts().sort_index().to_dict()
    print(f"   {week}주차: {dist}")

# 3·4주차에 공휴일 있는 영화 수
has_3w_holiday = (analysis['auto_3주차_공휴일'] > 0).sum()
has_4w_holiday = (analysis['auto_4주차_공휴일'] > 0).sum()
print(f"\n   3주차에 공휴일 있는 영화: {has_3w_holiday}편")
print(f"   4주차에 공휴일 있는 영화: {has_4w_holiday}편")


# ============================================================
# [7] 최종 변수 만들기 (셀 2에서 사용할 변수명)
# ============================================================
analysis['3주차_공휴일'] = analysis['auto_3주차_공휴일']
analysis['4주차_공휴일'] = analysis['auto_4주차_공휴일']

# 임시 컬럼 정리
analysis = analysis.drop(columns=['1주차_금요일',
                                    'auto_1주차_공휴일', 'auto_2주차_공휴일',
                                    'auto_3주차_공휴일', 'auto_4주차_공휴일'])

print(f"\n 최종 변수 추가 완료: '3주차_공휴일', '4주차_공휴일'")
print(f"   다음 셀(셀 2)에서 이 'analysis' DataFrame을 사용합니다.")

 한국 공휴일 로드: 77일
   2024년 임시공휴일 추가: ['제22대 국회의원 선거', '국군의 날 임시공휴일']
 자동 계산 완료: 1~4주차 공휴일

 검증: 자동 계산 vs 수동 계산 비교

1주차 일치: 131/131 (100.0%)
2주차 일치: 131/131 (100.0%)

 1·2주차 모두 100% 일치 → 자동 계산 신뢰 가능
   3·4주차도 자동 계산 결과를 사용합니다.

 자동 계산 결과 분포:
   1주차: {0: 119, 1: 6, 2: 6}
   2주차: {0: 120, 1: 7, 2: 2, 3: 2}
   3주차: {0: 125, 1: 3, 2: 2, 3: 1}
   4주차: {0: 123, 1: 6, 2: 2}

   3주차에 공휴일 있는 영화: 6편
   4주차에 공휴일 있는 영화: 8편

 최종 변수 추가 완료: '3주차_공휴일', '4주차_공휴일'
   다음 셀(셀 2)에서 이 'analysis' DataFrame을 사용합니다.


In [3]:
# ============================================================
# 셀 2. 6개 데이터에서 핵심 변수 추출
# ============================================================

# ============================================================
# [1] KOFIC 패널 → 1~4주차 주말관객 추출 (★ 새로 추가)
# ============================================================
panel = pd.read_excel(kofic_panel_file)
panel['개봉일'] = pd.to_datetime(panel['개봉일'].astype(str), format='%Y%m%d')

# 1~4주차 관객 추출 로직
# 기준주차==1 → 현재주=1주차, 다음주=2주차
# 기준주차==2 → 현재주=2주차, 다음주=3주차
# 기준주차==3 → 현재주=3주차, 다음주=4주차

# 각 영화의 1주차 = 기준주차==1 의 현재주_주말관객
week1 = panel[panel['기준주차']==1][['영화명','현재주_주말관객']].rename(
    columns={'현재주_주말관객':'1주차_주말관객'})
# 2주차 = 기준주차==1의 다음주_주말관객 (또는 기준주차==2의 현재주)
week2 = panel[panel['기준주차']==1][['영화명','다음주_주말관객']].rename(
    columns={'다음주_주말관객':'2주차_주말관객'})
# 3주차 = 기준주차==2의 다음주_주말관객
week3 = panel[panel['기준주차']==2][['영화명','다음주_주말관객']].rename(
    columns={'다음주_주말관객':'3주차_주말관객'})
# 4주차 = 기준주차==3의 다음주_주말관객
week4 = panel[panel['기준주차']==3][['영화명','다음주_주말관객']].rename(
    columns={'다음주_주말관객':'4주차_주말관객'})

# 합치기
weeks = week1.merge(week2, on='영화명', how='left')
weeks = weeks.merge(week3, on='영화명', how='left')
weeks = weeks.merge(week4, on='영화명', how='left')

print(f" 1~4주차 관객 추출: {len(weeks)}편")
print(f"   3주차 결측: {weeks['3주차_주말관객'].isnull().sum()}편")
print(f"   4주차 결측: {weeks['4주차_주말관객'].isnull().sum()}편")


# ============================================================
# [2] 분석파일에서 통제변수 추출 (3·4주차 공휴일 추가)
# ============================================================
#  변경: analysis는 셀 1.5에서 이미 만들었으니 그대로 사용
# (이전에 있던 pd.read_excel 줄은 제거)

control = analysis[[
    '영화명', '체급', '개봉일', '개봉요일', '누적관객수',
    '총 스크린수', '드롭률(%)',
    '1주차_공휴일수_주말', '2주차_공휴일수_주말',
    '3주차_공휴일', '4주차_공휴일',  #  추가
    '공휴일_차이(2-1)', '분류'
]].copy()

control = control.rename(columns={
    '드롭률(%)': '드롭률_1to2',
    '총 스크린수': '총_스크린수',
    '1주차_공휴일수_주말': '1주차_공휴일',
    '2주차_공휴일수_주말': '2주차_공휴일',
    '공휴일_차이(2-1)': '공휴일_차이'
})

control = control[control['분류'] != '데이터_부족'].reset_index(drop=True)
print(f" 분석 파일 처리: {len(control)}편")


# ============================================================
# [3] 타겟 변수 생성
# ============================================================
control['생존_여부'] = (control['드롭률_1to2'] < 50).astype(int)
print(f" 타겟: 생존 {control['생존_여부'].sum()}편 / 절벽 {(control['생존_여부']==0).sum()}편")


# ============================================================
# [4] 1~4주차 관객 머지 + 흥행 지속성 변수 생성
# ============================================================
control = control.merge(weeks, on='영화명', how='left')

# 4주차 생존비율 = 4주차 / 1주차 (4주차 NaN이면 0으로 처리)
control['4주차_생존비율'] = np.where(
    control['1주차_주말관객'] > 0,
    control['4주차_주말관객'].fillna(0) / control['1주차_주말관객'],
    0
)

# log 변환 (모델 입력용)
for col in ['1주차_주말관객','2주차_주말관객','3주차_주말관객','4주차_주말관객']:
    log_col = f'log_{col}'
    control[log_col] = np.log10(control[col].fillna(0).replace(0, np.nan))
    control[log_col] = control[log_col].fillna(0)

print(f" 흥행 지속성 변수 생성")
print(f"   4주차 생존비율 평균: {control['4주차_생존비율'].mean():.3f}")


# ============================================================
# [5] 단독·기대·검증 검색 처리 (이전과 동일)
# ============================================================
# 단독
solo_raw = pd.read_excel(solo_file)
solo_pivot = solo_raw.pivot_table(
    index='영화명', columns='시점', values='검색지수', aggfunc='mean'
).reset_index()
solo_pivot.columns.name = None
solo_pivot = solo_pivot.rename(columns={'개봉전':'단독_사전평균','개봉후':'단독_사후평균'})
solo_pivot['단독_사후사전비율'] = (
    solo_pivot['단독_사후평균'] / solo_pivot['단독_사전평균'].replace(0, np.nan)
)
solo_features = solo_pivot[['영화명','단독_사전평균','단독_사후평균','단독_사후사전비율']]

# 기대
expect_raw = pd.read_excel(expect_file, sheet_name='기대_검색량')
expect_features = expect_raw.groupby('영화명')['기대_검색지수'].mean().reset_index()
expect_features.columns = ['영화명','기대_강도']

# 검증
verif_raw = pd.read_excel(verif_file, sheet_name='검증_검색량')
verif_strength = verif_raw.groupby('영화명')['검증_검색지수'].mean().reset_index()
verif_strength.columns = ['영화명','검증_강도']

verif_raw['날짜'] = pd.to_datetime(verif_raw['날짜'])
verif_raw['개봉일'] = pd.to_datetime(verif_raw['개봉일'])
verif_raw['개봉_경과일'] = (verif_raw['날짜']-verif_raw['개봉일']).dt.days
day0 = verif_raw[verif_raw['개봉_경과일']==0].groupby('영화명')['검증_검색지수'].mean().reset_index()
day0.columns = ['영화명','_개봉일']
late = verif_raw[verif_raw['개봉_경과일'].between(5,7)].groupby('영화명')['검증_검색지수'].mean().reset_index()
late.columns = ['영화명','_후반']
verif_persist = day0.merge(late, on='영화명', how='left')
verif_persist['검증_지속력'] = verif_persist['_후반']/verif_persist['_개봉일'].replace(0, np.nan)
verif_features = verif_strength.merge(verif_persist[['영화명','검증_지속력']], on='영화명', how='left')

# 유튜브
yt_raw = pd.read_excel(youtube_file)
yt_features = pd.DataFrame()
yt_features['영화명'] = yt_raw['영화명']
yt_features['log_조회수'] = np.log10(yt_raw['조회수'].replace(0, 1))
yt_features['일평균_조회수'] = yt_raw['일평균_조회수']
yt_features['좋아요_비율'] = yt_raw['좋아요_비율']
yt_features['댓글_비율'] = yt_raw['댓글_비율']

print(f" 단독: {len(solo_features)}편 / 기대: {len(expect_features)}편 / 검증: {len(verif_features)}편 / 유튜브: {len(yt_features)}편")
print("\n다음 셀로 진행하세요.")

 1~4주차 관객 추출: 129편
   3주차 결측: 0편
   4주차 결측: 2편
 분석 파일 처리: 130편
 타겟: 생존 86편 / 절벽 44편
 흥행 지속성 변수 생성
   4주차 생존비율 평균: 0.220
 단독: 130편 / 기대: 125편 / 검증: 127편 / 유튜브: 130편

다음 셀로 진행하세요.


## 3️. 통합 + 핵심 파생 변수 + 저장

###  작업 내용
앞에서 추출한 변수들을 **영화명을 키로 머지**하여 마스터 테이블을 만듭니다.

###  핵심 파생 변수: 검증기대비율

```
검증기대비율 = 검증_강도 ÷ 기대_강도
```

| 비율 값 | 해석 |
|---|---|
| > 1 | 본 사람의 평가 검색이 사전 기대 검색보다 강함 → **입소문 흥행** |
| ≈ 1 | 기대만큼 검증됨 → 평이한 영화 |
| < 1 | 기대보다 검증이 약함 → **흥행 약화 위험** |

이 변수가 본 분석의 **숨은 황금 변수**가 될 가능성이 높습니다.

###  최종 마스터 테이블 구성 (33개 변수)

| 그룹 | 변수 수 | 변수 |
|---|---|---|
| 식별 | 5 | 영화명, 체급, 개봉일, 개봉요일, 누적관객수 |
| 흥행 타겟 | 3 | 드롭률_1to2, 생존_여부, 분류 |
| 1~4주차 관객 + 파생 | 9 | 1~4주차_주말관객, log 변환 4개, 4주차_생존비율 |
| 통제변수 | 4 | 총_스크린수, 1주차_공휴일, 2주차_공휴일, 공휴일_차이 |
| 네이버 단독 | 3 | 단독_사전평균, 단독_사후평균, 단독_사후사전비율 |
| 네이버 기대 | 1 | 기대_강도 |
| 네이버 검증 | 2 | 검증_강도, 검증_지속력 |
| 핵심 파생 | 1 | **검증기대비율**  |
| 유튜브 | 4 | log_조회수, 일평균_조회수, 좋아요_비율, 댓글_비율 |

###  출력 파일
- `master_dataset_전체.xlsx`: 130편 전체 (체급 비교용)
- `master_dataset_중형.xlsx`: 중형 영화 102편만 (본 분석 메인 표본)

In [ ]:
# 통제변수 (★)
'총_스크린수', '1주차_공휴일', '2주차_공휴일',
'3주차_공휴일', '4주차_공휴일',  #  추가
'공휴일_차이',

In [4]:
# ============================================================
# 셀 3. 통합 + 핵심 파생 변수 + 저장
# ============================================================

# control을 베이스로 left join (control이 가장 정확한 흥행 정보)
master = control.copy()
master = master.merge(solo_features, on='영화명', how='left')
master = master.merge(expect_features, on='영화명', how='left')
master = master.merge(verif_features, on='영화명', how='left')
master = master.merge(yt_features, on='영화명', how='left')

# 결측치 0으로 (검증·기대 데이터 없는 영화)
fill_zero_cols = [
    '단독_사전평균','단독_사후평균','단독_사후사전비율',
    '기대_강도','검증_강도','검증_지속력'
]
master[fill_zero_cols] = master[fill_zero_cols].fillna(0)

#  핵심 파생 변수: 검증/기대 비율
master['검증기대비율'] = np.where(
    master['기대_강도']>0,
    master['검증_강도']/master['기대_강도'],
    0
)

# 컬럼 순서 정리
final_columns = [
    # 식별
    '영화명','체급','개봉일','개봉요일','누적관객수',
    # 흥행 결과 (타겟)
    '드롭률_1to2','생존_여부','분류',
    # 1~4주차 관객 (★)
    '1주차_주말관객','2주차_주말관객','3주차_주말관객','4주차_주말관객',
    'log_1주차_주말관객','log_2주차_주말관객','log_3주차_주말관객','log_4주차_주말관객',
    '4주차_생존비율',
    # 통제변수
    '총_스크린수', '1주차_공휴일', '2주차_공휴일', '3주차_공휴일', '4주차_공휴일',
    # 단독 검색
    '단독_사전평균','단독_사후평균','단독_사후사전비율',
    # 기대 검색
    '기대_강도',
    # 검증 검색
    '검증_강도','검증_지속력',
    # 핵심 파생
    '검증기대비율',
    # 유튜브
    'log_조회수','일평균_조회수','좋아요_비율','댓글_비율'
]

master = master[final_columns]

# ============================================================
# 검증 출력
# ============================================================
print(f"{'='*60}")
print(f" 마스터 테이블 최종 검증")
print(f"{'='*60}")
print(f"   행: {len(master)}편")
print(f"   열: {len(master.columns)}개")
print(f"   결측치: {master.isnull().sum().sum()}개 (3·4주차 관객 NaN은 정상)")
print(f"\n   체급 분포: {master['체급'].value_counts().to_dict()}")
print(f"   분류 분포: {master['분류'].value_counts().to_dict()}")
print(f"   생존: {master['생존_여부'].sum()}편 / 절벽: {(master['생존_여부']==0).sum()}편")

print(f"\n 컬럼 목록 ({len(master.columns)}개):")
for i, col in enumerate(master.columns, 1):
    print(f"   {i:2d}. {col}")

# ============================================================
# 저장 (130편 + 중형만)
# ============================================================
master.to_excel('master_dataset_전체.xlsx', index=False)
print(f"\n 저장 1: master_dataset_전체.xlsx ({len(master)}편)")

master_mid = master[master['체급']=='중형'].reset_index(drop=True)
master_mid.to_excel('master_dataset_중형.xlsx', index=False)
print(f" 저장 2: master_dataset_중형.xlsx ({len(master_mid)}편)")

print(f"\n 자동 다운로드 시작...")
files.download('master_dataset_전체.xlsx')
files.download('master_dataset_중형.xlsx')

 마스터 테이블 최종 검증
   행: 130편
   열: 33개
   결측치: 46개 (3·4주차 관객 NaN은 정상)

   체급 분포: {'중형': 103, '중대형': 14, '대형': 13}
   분류 분포: {'정상_드롭': 86, '정상_드롭_1주차부스트': 26, '순수_역주행': 13, '공휴일_역주행': 5}
   생존: 86편 / 절벽: 44편

 컬럼 목록 (33개):
    1. 영화명
    2. 체급
    3. 개봉일
    4. 개봉요일
    5. 누적관객수
    6. 드롭률_1to2
    7. 생존_여부
    8. 분류
    9. 1주차_주말관객
   10. 2주차_주말관객
   11. 3주차_주말관객
   12. 4주차_주말관객
   13. log_1주차_주말관객
   14. log_2주차_주말관객
   15. log_3주차_주말관객
   16. log_4주차_주말관객
   17. 4주차_생존비율
   18. 총_스크린수
   19. 1주차_공휴일
   20. 2주차_공휴일
   21. 3주차_공휴일
   22. 4주차_공휴일
   23. 단독_사전평균
   24. 단독_사후평균
   25. 단독_사후사전비율
   26. 기대_강도
   27. 검증_강도
   28. 검증_지속력
   29. 검증기대비율
   30. log_조회수
   31. 일평균_조회수
   32. 좋아요_비율
   33. 댓글_비율

 저장 1: master_dataset_전체.xlsx (130편)
 저장 2: master_dataset_중형.xlsx (103편)

 자동 다운로드 시작...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##  데이터 통합 완료

### 다음 단계 (별도 노트북)

```
[다음 노트북 1] EDA — 시각화로 패턴 탐색
   - 체급별 평균 드롭률 막대그래프
   - 검증기대비율 vs 드롭률 산점도
   - 생존 vs 절벽 그룹의 디지털 반응 패턴 비교
   - 변수 간 상관관계 히트맵

[다음 노트북 2] 가설 검정
   - H1: 체급별 드롭률 차이 (ANOVA + Tukey HSD)
   - H2: 디지털 반응 ↔ 생존 (상관 + 로지스틱 회귀)
   - H3: 네이버 단일 vs 멀티채널 모델 (AUC 비교)

[다음 노트북 3] 모델링
   - 로지스틱 회귀 (베이스라인)
   - 랜덤 포레스트
   - XGBoost
   - 5-fold 교차검증으로 성능 비교
```

###  마스터 테이블 활용 가이드

```python
import pandas as pd
master = pd.read_excel('master_dataset_전체.xlsx')

# 메인 분석은 중형만
mid = pd.read_excel('master_dataset_중형.xlsx')

# 타겟 변수
y = mid['생존_여부']

# 입력 변수
X = mid[['검증기대비율', '검증_강도', '기대_강도', '단독_사후사전비율',
         'log_조회수', '좋아요_비율', '댓글_비율',
         '총_스크린수', '공휴일_차이']]
```